In [1]:
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

ModuleNotFoundError: No module named 'google'

In [ ]:
%cd '/content/gdrive/MyDrive/Documents/Projects/FCCATIC'

/content/gdrive/MyDrive/Documents/Projects/FCCATIC


In [ ]:
import pandas as pd
import warnings
import os
import numpy as np

warnings.filterwarnings('ignore')
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

In [ ]:
df = pd.read_csv("bankStatement.csv")

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['balance'] = (np.where(df['transaction_type'] == 'Spending', -1, 1) * df['amount']).cumsum()
df['spending'] = (np.where(df['transaction_type'] == 'Spending', -1, 0) * df['amount']).cumsum()
df['balance'] += 800

In [ ]:
balance = df.groupby(['date']).agg(
     PredictedBalance = ('balance','last'),
     Categories = ('category', lambda x: ', '.join(x)),
     Transactions = ('name', lambda x: ', '.join(x))
     ).reset_index()

In [ ]:
spending = df.copy()[df.transaction_type == 'Spending']
spending = spending.groupby(['date']).agg(
     balance = ('balance','last'),
     spending = ('spending', 'last'),
     categories = ('category', lambda x: ', '.join(x)),
     transactions = ('name', lambda x: ', '.join(x))
     ).reset_index()

In [ ]:
income = df.copy()[df.transaction_type != 'Spending']
income['income'] = income['amount'].cumsum()

income = income.groupby(['date']).agg(
     balance = ('balance','last'),
     income = ('income', 'last'),
     categories = ('category', lambda x: ', '.join(x)),
     transactions = ('name', lambda x: ', '.join(x))
     ).reset_index()

In [ ]:
rec_income = df.copy()[df.transaction_type != 'Spending']
rec_income = rec_income[rec_income.Type != 'non']
rec_income['income'] = rec_income['amount'].cumsum()

rec_income = rec_income.groupby(['date']).agg(
     balance = ('balance','last'),
     income = ('income', 'last'),
     amount = ('amount', sum),
     categories = ('category', lambda x: ', '.join(x)),
     transactions = ('name', lambda x: ', '.join(x))
     ).reset_index()

In [ ]:
bl = pd.read_csv("bankStatement_current.csv")
bl['date'] = pd.to_datetime(bl['date'])
bl['balance'] = (np.where(bl['transaction_type'] == 'Spending', -1, 1) * bl['amount']).cumsum()
bl['spending'] = (np.where(bl['transaction_type'] == 'Spending', -1, 0) * bl['amount']).cumsum()
bl['balance'] += 500

In [ ]:
curr_balance = bl.groupby(['date']).agg(
     balance = ('balance','last'),
     categories = ('category', lambda x: ', '.join(x)),
     transactions = ('name', lambda x: ', '.join(x))
     ).reset_index()

curr_spending = bl.copy()[bl.transaction_type == 'Spending']
curr_spending = curr_spending.groupby(['date']).agg(
     balance = ('balance','last'),
     spending = ('spending', 'last'),
     categories = ('category', lambda x: ', '.join(x)),
     transactions = ('name', lambda x: ', '.join(x))
     ).reset_index()


In [ ]:
import plotly.express as px

fig = px.line(balance, x='date', y='PredictedBalance', hover_data=['Transactions'],
                labels={ "PredictedBalance": "Amount", "date": "Time" },
                 title="Overdraft Warning Using Predictive Insights")
fig.add_scatter(x=spending['date'], y=spending['spending'], name='Predicted Spending')
fig.add_scatter(x=rec_income['date'], y=rec_income['amount'], mode='markers', name='Historic Income')
fig.add_scatter(x=curr_balance['date'], y=curr_balance['balance'], mode='lines+markers', name='Current Balance')
fig.add_scatter(x=curr_spending['date'], y=curr_spending['spending'], mode='lines+markers', name='Current Spending')
fig.update_layout(hovermode="x unified")